## **AI-powered Dataset Generator**

Generate datasets related to the T-shirt shop.

Select the dataset required from the options available: T-shirts, Sales or Inventory.

Set the quantity of required records. Default is 20 records.

Optionally, set the preferred LLM model: OpenAi/GPT, LLAMA (Huggingface), Claude or Gemini. Defualt is OpenAi/GPT.





In [ ]:
# pip install
!pip install torch bitsandbytes transformers openai anthropic gradio

In [ ]:
# Imports
import pandas as pd
import gradio as gr
from io import StringIO
import json

# models imports
from openai import OpenAI
import google.generativeai
import anthropic

# google collab
from google.colab import userdata # driver

# huggingface
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

In [ ]:
# templates
primary_key = {
    "T-shirts": "tshirt_id",
    "Sales": "sale_id",
    "Inventory": "tshirt_id",
}

tshirt_cols = """
    tshirt_id (string: unique T-shirt ID - Unique barcode)
    category (string: random category - Messages, Comics,..)
    color (string: Color of t-shirt)
    size (string: Size of the t-shirt - xs, s, m, l, xl, xxl)
    price (float: Price with 2 decimal places, example: 9.99 - The total price of the sale)
"""

sales_cols = """
    sale_id (string: Invoice Number - Unique for each sale transaction)
    date (date: date of the transaction)
    tshirt_id (string: T-shirt ID)
    quantity_sold (integer: Quantity Sold - The number of items sold)
    total_price (float: Total Price with 2 decimal places, example: 9.99 - The total price of the sale)
"""

inventory_cols = """
    tshirt_id (string: unique T-shrit ID)
    quantity_in_stock (integer: random Quantity in Stock - The current available quantity)
    reorder_level (integer: Reorder Level (like 70)- To alert branches when stock is low)
"""

columns = {
    "T-shirts": tshirt_cols,
    "Sales": sales_cols,
    "Inventory": inventory_cols,
}

In [ ]:
# Authentication

hf_token = userdata.get("HF_TOKEN")
openai_api_key = userdata.get("OPENAI_API_KEY")
anthropic_api_key = userdata.get('ANTHROPIC_API_KEY')
google_api_key = userdata.get('GOOGLE_API_KEY')

if not hf_token:
 raise ValueError("Missing HF_TOKEN. Set it as environment variables.")

if not openai_api_key:
 raise ValueError("Missing OPENAI_API_KEY. Set it as environment variables.")

if not anthropic_api_key:
 raise ValueError("Missing ANTHROPIC_API_KEY. Set it as environment variables.")

if not google_api_key:
 raise ValueError("Missing GOOGLE_API_KEY. Set it as environment variables.")

login(hf_token, add_to_git_credential=True)

In [ ]:
def get_llama_client(DEVICE):
  # quantization
  quant_config = BitsAndBytesConfig(
      load_in_4bit=True,
      bnb_4bit_use_double_quant=True,
      bnb_4bit_compute_dtype=torch.bfloat16,
      bnb_4bit_quant_type="nf4"
  )

  client = AutoModelForCausalLM.from_pretrained(MODELS['LLAMA'], device_map=DEVICE, quantization_config=quant_config)

  return client

In [ ]:
# constants
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODELS = {
          'GPT': 'gpt-4o-mini',
          'LLAMA': 'meta-llama/Meta-Llama-3.1-8B-Instruct',
          'CLAUDE': 'claude-3-haiku-20240307',
          'GEMINI': 'gemini-2.0-flash-exp'
         }

CLIENTS = {
            'GPT': OpenAI(api_key=openai_api_key),
            'LLAMA': get_llama_client(DEVICE=DEVICE),
            'CLAUDE': anthropic.Anthropic(api_key=anthropic_api_key),
            'GEMINI': OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key=google_api_key)
          }

In [ ]:
# clean model response from unrequired verbose
def clean_response(response):
    # remove trailing verbose
    processed_response = response.rsplit("```", 1)[0].strip()

    # remove leading verbose
    if len(processed_response.rsplit("```csv", 1)) > 1:
        processed_response = processed_response.rsplit("```csv", 1)[1].strip()
    else:
        start = processed_response.find(f'{primary_key[data_type]},')
        processed_response = processed_response[start:]

    return processed_response

In [ ]:
# get LLM response
def get_model_response(model, messages):

    # using LlaMa in Huggingface
    if model == "LLAMA":
      # setup tokenizer
      tokenizer = AutoTokenizer.from_pretrained(MODELS['LLAMA'])
      tokenizer.pad_token = tokenizer.eos_token
      # # modify prompt for LlaMa
      prompt = f'<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n{messages[1]["content"]}<|eot_id|>\n<|start_header_id|>assistant<|end_header_id|>\n'
      # tokenize input
      inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
      # get response tokens
      outputs = CLIENTS[model].generate(**inputs, max_new_tokens=1000)
      # get readable response
      response = tokenizer.decode(outputs[0], skip_special_tokens=True)

      return clean_response(response)

    # Claude has not adopted OpenAi's format!
    elif model == "CLAUDE":
        response = CLIENTS[model].messages.create(
                model=MODELS[model],
                max_tokens=600,
                system=messages[0]['content'],
                messages=messages[1:], # Claude takes the system prompt separately, thus removing it from the new list (shallow copy as in .copy())
            )

        return clean_response(response.content[0].text)
    else:
        response = CLIENTS[model].chat.completions.create(
                model=MODELS[model],
                # max_tokens=200,
                messages=messages,
                # response_format={"type": "json_object"}
            )

        return clean_response(response.choices[0].message.content)

In [ ]:
# Generate required dataset
def generate(data_type, qty, model, is_test=False):

    status = """
                <span>✅ Success! Please take a look at the generated CSV and dataset below.</span>
            """

    # test
    if is_test:

        csv = datasets[data_type]

    else:

        # set prompts - one-shot example required for Llama and Deepseek
        system_prompt = """
                You generate structured CSV data according to the user's requirements.
                The user will provide the list's topic, the required columns, and the quantity of the necessary records.
                Ensure the CSV format is valid, with proper headers and comma separation.
                Your output must be:

                ```csv
                header1,header2,...
                value1,value2,...
                ```
            """


        user_prompt = f"""
                Generate a CSV with {qty} rows of synthetic data. Columns {columns[data_type]}.
                Use the columns' name as the header in the first row of the generated dataset.
                The first column must be: {primary_key[data_type]}.
                Ensure the CSV format is valid, with proper headers and comma separation.
                Do not add any explanation just return the data inside triple backticks like this:

                ```csv
                header1,header2,...
                value1,value2,...
                ```
            """

        # print(f'user_prompt: {user_prompt}')

        messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
               ]

        csv = get_model_response(model=model, messages=messages)

        # print(f'after AI: {csv}')


    # Convert response to DataFrame
    try:
        # table dataframe
        table = pd.read_csv(StringIO(csv.strip()))

        # remove leading/trailing spaces if any
        # columns
        table.columns = table.columns.str.strip()

        # cells
        table = table.map(lambda x: x.strip() if isinstance(x, str) else x)

        # Format the float columns if it exists - Pandas removes trailing zeros for float values!
        for column in table.columns:
            if column == 'total_price' or column == 'price':
                table[column] = table[column].map(lambda x: f"{x:.2f}")
                table[column] = pd.to_numeric(table[column], errors="coerce")

        # convert to json
        indexed_table = table.set_index(primary_key[data_type]) # dataframe
        df_to_dict = indexed_table.to_dict(orient="index")
        json_output = json.dumps(df_to_dict, indent=4)

    except Exception as e:
        return None, csv, None, pd.DataFrame(), f"<span style='color:red;'>⚠️ Error parsing dataset: {str(e)}</span>", None

    return gr.update(visible=True), csv, json_output, table, status, gr.update(interactive=True, visible=True)

In [ ]:
# Update UI after any change in 'Required Dataset' section
def get_ready(data_type):

    if data_type is None:
        status = """
                    <span>ℹ️ Please select <b>Required Dataset</b> above to proceed.
                    You may also change the <b>Number of Records</b> to be generated.</span>
                """
        return None, status

    status = """
                <span>ℹ️ When ready, click on <b>Generate Dataset</b> to generate the required quantity of records. </span>
            """
    btn = gr.update(value="Generate Dataset", interactive=True, visible=True)

    return btn, status

In [ ]:
# reset UI to defaults
def reset():
    selected_model = gr.update(value=list(MODELS.keys())[0])

    data_type  = gr.update(value=None)

    qty_records = gr.update(value=20)

    csv_data = gr.update(value="")

    json_data = gr.update(value="")

    table_data = gr.update(value=pd.DataFrame())

    data_panel = gr.update(visible=False)

    status_bar = """
                    <span>ℹ️ Please select <b>Required Dataset</b> above to proceed.
                    You may also change the <b>Number of Records</b> to be generated.</span>
                """

    generate_btn = gr.update(interactive=False, visible=False)

    reset_btn = gr.update(interactive=False, visible=False)

    return selected_model, data_type, qty_records, csv_data, json_data, table_data, data_panel, status_bar, generate_btn, reset_btn

In [ ]:
# UI
with gr.Blocks() as demo:
    gr.Markdown("## AI-powered Dataset Generator")

    with gr.Accordion("(Optional) Model Selection", open=False):
         # Input data required
         exclude_keys = {} # add keys to exclude if necessary like: {'LLAMA', 'DEEPSEEK'}
         models_list = [key for key in MODELS if key not in exclude_keys]
         selected_model = gr.Radio(
            models_list,
            value=models_list[0],
            label="LLM Models Available",
            info="Select the LLM to be used to generate the dataset."
        )
    with gr.Row(equal_height=True):
        with gr.Column():
             # Input data required
             data_type = gr.Radio(
                ["T-shirts", "Sales", "Inventory"],
                value=None,
                label="Required Dataset",
                info="Select the type of data required."
            )
        with gr.Column():
             # Input quantity of records
             qty_records = gr.Slider(
                 minimum=10,
                 maximum=50,
                 step=10,
                 value=20,
                 label="Number of Records",
                 info="Select the quantity of records to generate.",
                 interactive=True,
                 show_reset_button=False)
    with gr.Row(): #
        generate_btn = gr.Button(value="Generate Data", elem_classes="btn_edit", interactive=False, visible=False)

        reset_btn = gr.Button(value="Reset", interactive=False, visible=False)
    with gr.Row():
        status_bar = gr.Markdown(
            """
            <span>ℹ️ Please select <b>Required Dataset</b> above to proceed.
            You may also change the <b>Number of Records</b> to be generated.</span>
            """
        )
    with gr.Row(equal_height=True): # visible=False
        with gr.Column(visible=False) as data_panel:
            with gr.Tab("CSV"):
                # Display generated CSV data
                csv_data = gr.Textbox(value="", label="Generated CSV Data", interactive=False, elem_classes=["full-height-tab"])
            with gr.Tab("JSON"):
                # Display generated JSON object
                json_data = gr.Textbox(value="", label="Generated JSON", interactive=False, elem_classes=["full-height-tab"])
            with gr.Tab("Dataset"):
                # Display generated Dataset
                table_data = gr.Dataframe(value=pd.DataFrame(), label="Generated Dataset", interactive=False)

    data_type.change(fn=get_ready, inputs=[data_type], outputs=[generate_btn, status_bar])

    generate_btn.click(fn=generate,
                       inputs=[data_type, qty_records, selected_model],
                       outputs=[data_panel, csv_data, json_data, table_data, status_bar, reset_btn])

    reset_btn.click(fn=reset, outputs=[selected_model,
                                       data_type,
                                       qty_records,
                                       csv_data,
                                       json_data,
                                       table_data,
                                       data_panel,
                                       status_bar,
                                       generate_btn,
                                       reset_btn])

    demo.css = """
        /* Change 'Generate Dataset' button colors */
        .btn_edit {
            background-color: green;
            color: white;
        }

        /* Set height of textarea components */
        .full-height-tab {
            height: 275px!important;
        }

        .full-height-tab textarea {
            height: 275px!important;
        }
    """

demo.launch()

## **Lessons learned**

The open source models require more detailed prompts that include (one-shot or few-shots) examples of the output required.

Likewise, the specific prompt for Llama including the special tokens, generated a better outcome.

Generally, all models included unrequired verbose in their response, even when when the prompt specifically specifically asked to avoid it.